In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

**Importing the Dataset**


In [ ]:
from google.colab import files
files.upload()

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d mohitsingh1804/plantvillage

Dataset URL: https://www.kaggle.com/datasets/mohitsingh1804/plantvillage
License(s): GPL-2.0
100% 818M/818M [00:04<00:00, 191MB/s]



In [ ]:
# Unzip your downloaded dataset
!unzip plantvillage.zip

# Optional: Remove the original zip file to save workspace space
!rm plantvillage.zip

**CNN Model**

In [6]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

In [7]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
NUM_CLASSES = 38

In [8]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

training_set = train_datagen.flow_from_directory(
    "PlantVillage/train",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

test_set = test_datagen.flow_from_directory(
    "PlantVillage/val",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

Found 43444 images belonging to 38 classes.
Found 10861 images belonging to 38 classes.


In [9]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [10]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

In [11]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),

    ModelCheckpoint(
        "best_resnet50.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [12]:
history = model.fit(
    training_set,
    validation_data=test_set,
    epochs=7,
    callbacks=callbacks
)

Epoch 1/7
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 422ms/step - accuracy: 0.8046 - loss: 0.7153
Epoch 1: val_accuracy improved from None to 0.94153, saving model to best_resnet50.keras

Epoch 1: finished saving model to best_resnet50.keras
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 627s 449ms/step - accuracy: 0.8844 - loss: 0.3865 - val_accuracy: 0.9415 - val_loss: 0.1783 - learning_rate: 0.0010
Epoch 2/7
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 405ms/step - accuracy: 0.9382 - loss: 0.1845
Epoch 2: val_accuracy improved from 0.94153 to 0.96280, saving model to best_resnet50.keras

Epoch 2: finished saving model to best_resnet50.keras
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 580s 427ms/step - accuracy: 0.9401 - loss: 0.1807 - val_accuracy: 0.9628 - val_loss: 0.1122 - learning_rate: 0.0010
Epoch 3/7
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 408ms/step - accuracy: 0.9465 - loss: 0.1622
Epoch 3: val_accuracy did not improve from 0.96280
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 581s 428ms/step - accuracy: 0.9480 - loss: 0.1593 - val_accuracy: 

In [13]:
base_model.trainable = True

for layer in base_model.layers[:-50]:
    layer.trainable = False

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history_fine = model.fit(
    training_set,
    validation_data=test_set,
    epochs=5,
    callbacks=callbacks
)

model.save("final_leaf_disease_model.keras")

Epoch 1/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.9188 - loss: 0.2857
Epoch 1: val_accuracy improved from 0.97578 to 0.97753, saving model to best_resnet50.keras

Epoch 1: finished saving model to best_resnet50.keras
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 643s 454ms/step - accuracy: 0.9507 - loss: 0.1575 - val_accuracy: 0.9775 - val_loss: 0.0692 - learning_rate: 1.0000e-05
Epoch 2/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 420ms/step - accuracy: 0.9771 - loss: 0.0694
Epoch 2: val_accuracy improved from 0.97753 to 0.97864, saving model to best_resnet50.keras

Epoch 2: finished saving model to best_resnet50.keras
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 600s 442ms/step - accuracy: 0.9789 - loss: 0.0630 - val_accuracy: 0.9786 - val_loss: 0.0639 - learning_rate: 1.0000e-05
Epoch 3/5
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - accuracy: 0.9869 - loss: 0.0435
Epoch 3: val_accuracy improved from 0.97864 to 0.98407, saving model to best_resnet50.keras

Epoch 3: finished saving model to best_resne